In [4]:
import requests
import pandas as pd

url = "https://api.open-meteo.com/v1/forecast?latitude=38.72&longitude=-9.14&daily=temperature_2m_max,temperature_2m_min,precipitation_sum&timezone=Europe%2FLisbon"
data = requests.get(url).json()
df = pd.DataFrame(data['daily'])
print(df.head())

         time  temperature_2m_max  temperature_2m_min  precipitation_sum
0  2026-11-15                19.1                15.8                2.4
1  2026-11-16                19.5                16.1                2.9
2  2026-11-17                21.3                18.3                1.7
3  2026-11-18                18.9                18.0                8.4
4  2026-11-19                18.7                12.5                0.0


In [5]:
# TRANSFORM 1 - Rename columns to clean names
df = df.rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max_c',
    'temperature_2m_min': 'temp_min_c',
    'precipitation_sum': 'rain_mm'
})

# TRANSFORM 2 - New column: temp range
df['temp_range_c'] = df['temp_max_c'] - df['temp_min_c']

print(df.head())

         date  temp_max_c  temp_min_c  rain_mm  temp_range_c
0  2026-11-15        19.1        15.8      2.4           3.3
1  2026-11-16        19.5        16.1      2.9           3.4
2  2026-11-17        21.3        18.3      1.7           3.0
3  2026-11-18        18.9        18.0      8.4           0.9
4  2026-11-19        18.7        12.5      0.0           6.2


In [6]:
# TRANSFORM 3 - Filter: only keep days with rain < 5mm
df_clean = df[df['rain_mm'] < 5].copy()

# TRANSFORM 4 - Sort: hottest day first
df_clean = df_clean.sort_values('temp_max_c', ascending=False)

print(df_clean)

# LOAD - Save to data/processed/output.csv
import os
os.makedirs('data/processed', exist_ok=True)
df_clean.to_csv('data/processed/output.csv', index=False)
print("\nSAVED!")

         date  temp_max_c  temp_min_c  rain_mm  temp_range_c
2  2026-11-17        21.3        18.3      1.7           3.0
1  2026-11-16        19.5        16.1      2.9           3.4
0  2026-11-15        19.1        15.8      2.4           3.3
4  2026-11-19        18.7        12.5      0.0           6.2
5  2026-11-20        17.0        10.5      0.0           6.5

SAVED!


Conclusion

1. Extract: I got 7 days of Lisbon weather from Open-Meteo API.
2. Transform: I renamed columns, created temp_range_c, filtered rain < 5mm, sorted by hottest.
3. Result: The hottest day was 2026-11-17 with 21.3°C.
4. The rainiest day 2026-11-18 with 8.4mm was removed by my filter.
5. Load: Saved 5 clean rows to data/processed/output.csv.